In [1]:
%%sql
SELECT * 
FROM bronze_nasa_nrt
LIMIT 10;

StatementMeta(, 1784ba0f-efdc-4718-860e-59210c9c507d, 4, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 17 fields>

In [23]:
%%sql
SELECT 
    acq_date,
    acq_time,
    confidence,
    frp AS potencia_fuego_mw,
    bright_ti4 AS temp_kelvin,
    latitude,
    longitude
FROM bronze_nasa_nrt
WHERE confidence IN ('nominal', 'high')
ORDER BY frp DESC
LIMIT 10;

StatementMeta(, b7a33626-ace4-42c0-9b4e-857d84b2cb1c, 27, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>

In [2]:
%%sql
SELECT 
    MIN(overallStartTime) AS fecha_inicio_minima,
    MAX(overallStartTime) AS fecha_inicio_maxima,
    min(ingestion_timestamp) as min_ingesta,
    max(ingestion_timestamp)as max_ingesta,
    COUNT(*) AS total_registros
FROM bronze_dgt_traffic;

StatementMeta(, 1784ba0f-efdc-4718-860e-59210c9c507d, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

In [9]:
%%sql
SELECT 
    DATE(ingestion_timestamp) AS fecha_ingesta,
    DATE(overallStartTime) AS fecha,
    COUNT(*) AS total_registros
FROM bronze_dgt_traffic
GROUP BY date(ingestion_timestamp), date(overallStartTime) 
ORDER BY fecha_ingesta DESC;

StatementMeta(, b7a33626-ace4-42c0-9b4e-857d84b2cb1c, 13, Finished, Available, Finished, False)

<Spark SQL result set with 161 rows and 3 fields>

In [19]:
%%sql
SELECT 
    *
FROM bronze_dgt_traffic
where updated_timestamp is not null;

StatementMeta(, b7a33626-ace4-42c0-9b4e-857d84b2cb1c, 23, Finished, Available, Finished, False)

<Spark SQL result set with 785 rows and 54 fields>

In [3]:
from pyspark.sql import functions as F

# Bounding box de control para verificar datos fuera de España + Búfer
LAT_MIN, LAT_MAX = 27.0, 44.0
LON_MIN, LON_MAX = -18.5, 5.0

StatementMeta(, 1784ba0f-efdc-4718-860e-59210c9c507d, 7, Finished, Available, Finished, False)

In [1]:
%%sql
select * from silver_weather limit 10;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 4, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 17 fields>

In [2]:
%%sql
SELECT DISTINCT latitude, longitude 
FROM bronze_weather 
WHERE landing_source_file LIKE '%134035%';

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [3]:
%%sql

SELECT count(*) from bronze_weather;
select count(*) from silver_weather;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 7, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [4]:
%%sql
-- 1. Verificación de Volumetría Total y Ventana de Predicción Temporal
SELECT 
    COUNT(1) AS total_registros,
    COUNT(DISTINCT CONCAT(latitude, '_', longitude)) AS total_nodos_geograficos,
    MIN(forecast_timestamp) AS primera_fecha_prediccion,
    MAX(forecast_timestamp) AS ultima_fecha_prediccion,
    DATEDIFF(day, MIN(forecast_timestamp), MAX(forecast_timestamp)) + 1 AS dias_cobertura
FROM silver_weather;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

In [5]:
%%sql
-- 2. Trazabilidad del Pipeline Medallion y Archivos Origen
SELECT 
    landing_source_file,
    MIN(ingestion_timestamp) AS fecha_primera_ingesta,
    MAX(ingestion_timestamp) AS fecha_ultima_ingesta,
    COUNT(1) AS registros_asociados
FROM silver_weather
GROUP BY landing_source_file
ORDER BY fecha_primera_ingesta DESC;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 9, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>

In [6]:
%%sql
-- 3. Control de Duplicados sobre la Clave Primaria (Latitud, Longitud, Timestamp Predicción)
SELECT 
    latitude, 
    longitude, 
    forecast_timestamp, 
    COUNT(1) AS apariciones
FROM silver_weather
GROUP BY latitude, longitude, forecast_timestamp
HAVING COUNT(1) > 1;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 4 fields>

In [7]:
%%sql
-- 4. Validación de Rangos Físicos Reales (Outliers / Sensores Defectuosos)
SELECT 
    SUM(CASE WHEN temperature_celsius < -30.0 OR temperature_celsius > 50.0 THEN 1 ELSE 0 END) AS temp_out_of_range,
    SUM(CASE WHEN humidity_percentage < 0 OR humidity_percentage > 100 THEN 1 ELSE 0 END) AS humidity_out_of_range,
    SUM(CASE WHEN precipitation_mm < 0.0 OR precipitation_mm > 300.0 THEN 1 ELSE 0 END) AS precip_out_of_range,
    SUM(CASE WHEN wind_speed_kmh < 0.0 OR wind_speed_kmh > 250.0 THEN 1 ELSE 0 END) AS wind_out_of_range
FROM silver_weather;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [8]:
%%sql
-- 5. Evaluación de Nulos en Variables Críticas de Modelado
SELECT 
    COUNT(1) AS total_filas,
    ROUND((1.0 - (COUNT(temperature_celsius) / CAST(COUNT(1) AS DOUBLE))) * 100, 4) AS pct_null_temperature,
    ROUND((1.0 - (COUNT(humidity_percentage) / CAST(COUNT(1) AS DOUBLE))) * 100, 4) AS pct_null_humidity,
    ROUND((1.0 - (COUNT(precipitation_mm) / CAST(COUNT(1) AS DOUBLE))) * 100, 4) AS pct_null_precipitation,
    ROUND((1.0 - (COUNT(wind_speed_kmh) / CAST(COUNT(1) AS DOUBLE))) * 100, 4) AS pct_null_wind
FROM silver_weather;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

In [9]:
%%sql
-- 6. Distribución de Lecturas por Nodo Geográfico
SELECT 
    latitude,
    longitude,
    elevation,
    timezone,
    COUNT(1) AS lecturas_horas_registradas,
    MIN(temperature_celsius) AS temp_minima,
    AVG(temperature_celsius) AS temp_media,
    MAX(temperature_celsius) AS temp_maxima
FROM silver_weather
GROUP BY latitude, longitude, elevation, timezone
ORDER BY latitude DESC, longitude ASC
LIMIT 10;

StatementMeta(, effa4024-2549-4162-a907-95236743e912, 13, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 8 fields>

In [10]:
%%sql
SELECT 
    hourly.time AS unidad_tiempo,
    hourly.temperature_2m AS unidad_temperatura,
    hourly.relative_humidity_2m AS unidad_humedad,
    hourly.precipitation AS unidad_precipitacion,
    hourly.rain AS unidad_lluvia,
    hourly.showers AS unidad_chubascos,
    hourly.snowfall AS unidad_nieve,
    hourly.wind_speed_10m AS unidad_velocidad_viento,
    hourly.wind_gusts_10m AS unidad_racha_viento
FROM bronze_weather
LIMIT 1;

StatementMeta(, 1784ba0f-efdc-4718-860e-59210c9c507d, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 9 fields>

In [6]:
%%sql
SELECT 
    COUNT(*) AS total_registros,
    COUNT(CASE WHEN latitude IS NULL OR longitude IS NULL THEN 1 END) AS coords_nulas,
    MIN(latitude) AS min_latitud,
    MAX(latitude) AS max_latitud,
    MIN(longitude) AS min_longitud,
    MAX(longitude) AS max_longitud,
    MIN(elevation) AS min_elevacion_m,
    MAX(elevation) AS max_elevacion_m
FROM bronze_weather;

StatementMeta(, 1784ba0f-efdc-4718-860e-59210c9c507d, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 8 fields>